# Imports

In [5]:
import torch

from src.data_loader import orchestrate_data_download
from src.data_preprocessor import orchestrate_data_preprocess
from src.data_collator import create_data_collator
from src.training_arguments import create_training_arguments
from src.model_loader import load_model
from src.model_trainer import create_trainer
from src.config import MODEL
from src.inference import orchestrate_evaluation
from transformers import set_seed

In [6]:
set_seed(42)

# Data download and save

In [7]:
dataset_dict = orchestrate_data_download()

Saving the dataset (0/1 shards):   0%|          | 0/1264 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1629 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/182 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/453 [00:00<?, ? examples/s]

Dataset dictionary: DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 1629
    })
    val: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 182
    })
    test: Dataset({
        features: ['text', 'label_text', 'label'],
        num_rows: 453
    })
})


# Data tokenization

In [8]:
tokenizer_distilbert = orchestrate_data_preprocess(model_name="distilbert")
tokenizer_distilbert.save_pretrained(MODEL["distilbert"]["save_tokenizer_path"])

collator_distilbert = create_data_collator(tokenizer=tokenizer_distilbert)

Map:   0%|          | 0/182 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1629 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/182 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/453 [00:00<?, ? examples/s]

Tokenized dataset dictionary: DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1629
    })
    val: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 182
    })
    test: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 453
    })
})


In [9]:
tokenizer_finbert = orchestrate_data_preprocess(model_name="finbert")
tokenizer_finbert.save_pretrained(MODEL["finbert"]["save_tokenizer_path"])

collator_finbert = create_data_collator(tokenizer=tokenizer_finbert)

Map:   0%|          | 0/182 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1629 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/182 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/453 [00:00<?, ? examples/s]

Tokenized dataset dictionary: DatasetDict({
    train: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1629
    })
    val: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 182
    })
    test: Dataset({
        features: ['text', 'label_text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 453
    })
})


# Training arguments

In [10]:
training_arguments_distilbert = create_training_arguments(output_path=MODEL["distilbert"]["save_fine_tuned_path"])

In [11]:
training_arguments_finbert = create_training_arguments(output_path=MODEL["finbert"]["save_fine_tuned_path"])

# Model loading

In [12]:
model_distilbert = load_model(
    checkpoint=MODEL["distilbert"]["checkpoint"], 
    num_labels=MODEL["distilbert"]["num_labels"], 
    device="cuda" if torch.cuda.is_available() else "cpu"
    )

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
model_finbert = load_model(
    checkpoint=MODEL["finbert"]["checkpoint"], 
    num_labels=MODEL["finbert"]["num_labels"], 
    device="cuda" if torch.cuda.is_available() else "cpu"
    )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Model training

In [14]:
trainer_distilbert = create_trainer(
    model=model_distilbert, 
    training_arguments=training_arguments_distilbert, 
    data_collator=collator_distilbert, 
    tokenized_datasets_path=MODEL["distilbert"]["save_tokenized_path"],
    tokenizer=tokenizer_distilbert
    )

trainer_distilbert.train()

trainer_distilbert.save_model(MODEL["distilbert"]["save_final_fine_tuned"])

Step,Training Loss,Validation Loss,Accuracy,F1
50,3.865173,0.751043,0.703297,0.458388
100,2.041369,0.428831,0.818681,0.706385
150,1.108888,0.242678,0.917582,0.906255
200,0.653516,0.110795,0.956044,0.949587
250,0.416781,0.136348,0.950549,0.919218
300,0.308103,0.105820,0.956044,0.938586


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [15]:
trainer_finbert = create_trainer(
    model=model_finbert, 
    training_arguments=training_arguments_finbert, 
    data_collator=collator_finbert, 
    tokenized_datasets_path=MODEL["finbert"]["save_tokenized_path"],
    tokenizer=tokenizer_finbert
    )

trainer_finbert.train()

trainer_finbert.save_model(MODEL["finbert"]["save_final_fine_tuned"])

Step,Training Loss,Validation Loss,Accuracy,F1
50,8.179491,0.639972,0.725275,0.577564
100,1.590276,0.278351,0.857143,0.763638
150,0.713505,0.082814,0.989011,0.980967
200,0.363296,0.112442,0.967033,0.955685
250,0.167070,0.058294,0.983516,0.977439


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# Evaluate

In [16]:
orchestrate_evaluation(model_name="distilbert", model=model_distilbert, data_collator=collator_distilbert)

{'accuracy': 0.9624724061810155, 'f1': 0.9484122492611456}


In [17]:
orchestrate_evaluation(model_name="finbert", model=model_finbert, data_collator=collator_finbert)

{'accuracy': 0.9602649006622517, 'f1': 0.9426610603644097}
